In [1]:
import numpy as np
import polars as pl
from pathlib import Path
import hashlib
import json
from sklearn.preprocessing import LabelEncoder

In [2]:
schema = {
    "section id": pl.Utf8,
    "recording id": pl.Utf8,

    "timestamp [ns]": pl.Int64,
    "gaze x [px]": pl.Float64,
    "gaze y [px]": pl.Float64,

    "fixation id": pl.Utf8,
    "blink id": pl.Utf8,
    "saccade id": pl.Utf8,

    "azimuth [deg]": pl.Float64,
    "elevation [deg]": pl.Float64,

    "start timestamp [ns]": pl.Int64,
    "end timestamp [ns]": pl.Int64,

    "duration [ms]": pl.Int64,
    "amplitude [px]": pl.Float64,
    "amplitude [deg]": pl.Float64,
    "mean velocity [px/s]": pl.Float64,
    "peak velocity [px/s]": pl.Float64,

    "timestamp [ms]": pl.Int64,

    "label": pl.Utf8,
    "norm_time": pl.Float64,
    "label_right": pl.Utf8,

    "duration [ms] fix": pl.Int64,
    "fixation x [px]": pl.Float64,
    "fixation y [px]": pl.Float64,

    "duration [ms] sacc": pl.Int64,
    "amplitude [deg] sacc": pl.Float64,
    "mean velocity [px/s] sacc": pl.Float64,
    "peak velocity [px/s] sacc": pl.Float64,

    "duration [ms] blink": pl.Int64,

    "gyro x [deg/s]": pl.Float64,
    "gyro y [deg/s]": pl.Float64,
    "gyro z [deg/s]": pl.Float64,

    "acceleration x [g]": pl.Float64,
    "acceleration y [g]": pl.Float64,
    "acceleration z [g]": pl.Float64,

    "roll [deg]": pl.Float64,
    "pitch [deg]": pl.Float64,
    "yaw [deg]": pl.Float64,

    "quaternion w": pl.Float64,
    "quaternion x": pl.Float64,
    "quaternion y": pl.Float64,
    "quaternion z": pl.Float64,

    "pupil diameter left [mm]": pl.Float64,
    "pupil diameter right [mm]": pl.Float64,

    "eyeball center left x [mm]": pl.Float64,
    "eyeball center left y [mm]": pl.Float64,
    "eyeball center left z [mm]": pl.Float64,

    "eyeball center right x [mm]": pl.Float64,
    "eyeball center right y [mm]": pl.Float64,
    "eyeball center right z [mm]": pl.Float64,

    "optical axis left x": pl.Float64,
    "optical axis left y": pl.Float64,
    "optical axis left z": pl.Float64,

    "optical axis right x": pl.Float64,
    "optical axis right y": pl.Float64,
    "optical axis right z": pl.Float64,

    "eyelid angle top left [rad]": pl.Float64,
    "eyelid angle bottom left [rad]": pl.Float64,
    "eyelid aperture left [mm]": pl.Float64,

    "eyelid angle top right [rad]": pl.Float64,
    "eyelid angle bottom right [rad]": pl.Float64,
    "eyelid aperture right [mm]": pl.Float64,
}

In [3]:
feature_cols = [
    "gaze x [px]",
    "gaze y [px]",
    "azimuth [deg]",
    "elevation [deg]",
    "pupil diameter left [mm]",
    "pupil diameter right [mm]",
    
    "optical axis left x",
    "optical axis left y",
    "optical axis left z",
    "optical axis right x",
    "optical axis right y",
    "optical axis right z",

    "gyro x [deg/s]",
    "gyro y [deg/s]",
    "gyro z [deg/s]",
    "acceleration x [g]",
    "acceleration y [g]",
    "acceleration z [g]",
    
    "roll [deg]",
    "pitch [deg]",
    "yaw [deg]",

    "fixation id",
    "blink id",
    "saccade id",
    "duration [ms]",
    "fixation x [px]",
    "fixation y [px]",
    "duration [ms] sacc",
    "amplitude [deg]",
    "mean velocity [px/s]",
    "peak velocity [px/s]",
    "duration [ms] blink",
    "quaternion w",
    "quaternion x",
    "quaternion y",
    "quaternion z",
    "eyeball center left x [mm]",
    "eyeball center left y [mm]",
    "eyeball center left z [mm]",
    "eyeball center right x [mm]",
    "eyeball center right y [mm]",
    "eyeball center right z [mm]",
    "eyelid angle top left [rad]",
    "eyelid angle bottom left [rad]",
    "eyelid aperture left [mm]",
    "eyelid angle top right [rad]",
    "eyelid angle bottom right [rad]",
    "eyelid aperture right [mm]",
]

In [4]:
def load_parquet(parquet_path):
    parquet_path = Path(parquet_path)

    if parquet_path.exists():
        print("Loading parquet...")
        return pl.scan_parquet(parquet_path)

    print("Parquet does not exist!!!!!!")
    return None

In [5]:
data_path = '../../data/'
training_data_path = data_path + "Final Training Data/"
merged_df = load_parquet(training_data_path + "merged_output.parquet")
to_label_df = pl.read_csv(data_path + "Final Data For Labeling/" + "merged_output.csv")

labels = merged_df.select("label").unique().collect().to_series().to_numpy()

Loading parquet...


In [6]:
print(merged_df.select(pl.col("recording id").value_counts()).collect())

for label in merged_df.select(pl.col("label").value_counts()).collect():
    print(label)

shape: (22, 1)
┌─────────────────────────────────┐
│ recording id                    │
│ ---                             │
│ struct[2]                       │
╞═════════════════════════════════╡
│ {"97b2cb35-e6ef-4ee4-b532-5d26… │
│ {"63efffc7-e347-47fb-9615-f7da… │
│ {"3b1c507b-c290-4250-95c9-e21d… │
│ {"f076d0a4-2afd-4c60-9883-4f64… │
│ {"0fc57dc8-de68-4b41-bf69-59a5… │
│ …                               │
│ {"383b0fe8-654a-4f4f-84cd-3754… │
│ {"911806b6-27bd-4b56-bd2f-45d9… │
│ {"e2e75728-4988-49c4-b2cc-9ec7… │
│ {"5bcd3401-e378-482f-b9c6-37ed… │
│ {"b636a895-09a2-4fe2-9c37-973e… │
└─────────────────────────────────┘
shape: (3,)
Series: 'label' [struct[2]]
[
	{"Assembling",255151}
	{"Looking at guide book",234385}
	{"Searching for pieces",190479}
]


In [7]:
le = LabelEncoder()
le.fit(labels)

label_map = dict(zip(le.classes_, range(len(le.classes_))))
    
merged_df = merged_df.with_columns(
    pl.col("label").replace_strict(label_map).alias("label")
)

In [8]:
def get_cache_dir(training_data_path, settings, feature_cols):
    config = {
        "window_size": settings["window size"],
        "overlap": settings["overlap"],
        "target_length": settings["target length"],
        "min_samples": settings["min samples"],
        "features": feature_cols,
        "use video min length": settings["use video min length"],
        "use engineered features": settings["use engineered features"],
        "use raw features": settings["use raw features"],
        "video percentage": settings["video percentage"],
        "video offset": settings["video offset"]
    }

    config_str = json.dumps(config, sort_keys=True).encode()
    config_hash = hashlib.md5(config_str).hexdigest()

    cache_dir = Path(training_data_path) / "Windowed Data" / config_hash
    cache_dir.mkdir(parents=True, exist_ok=True)

    return cache_dir

In [9]:
def crop_video_percentage(df: pl.DataFrame, percentage: float, offset: float):
    if percentage >= 1.0 and offset == 0.0:
        return df

    if not (0 <= offset <= 1):
        raise ValueError("offset must be between 0 and 1")

    if not (0 < percentage <= 1):
        raise ValueError("percentage must be in (0, 1]")

    if offset + percentage > 1:
        raise ValueError("offset + percentage cannot exceed 1")

    t0 = df["timestamp [ns]"][0]
    t1 = df["timestamp [ns]"][-1]
    
    duration = t1 - t0
    
    start = t0 + offset * duration
    end = start + percentage * duration
 
    return df.filter(
        (pl.col("timestamp [ns]") >= start) &
        (pl.col("timestamp [ns]") < end)
    )

In [10]:
def build_cache(subjects, merged_df, feature_cols, settings, cache_dir):

    if settings["use video min length"]:
        subject_lengths = compute_subject_lengths(subjects, merged_df)
        min_len = min(subject_lengths.values())

        print("Shortest subject length:", min_len)

    cache_dir = Path(cache_dir)
    with open(cache_dir / "settings.json", "w", encoding="utf-8") as f:
        json.dump(settings, f, indent=4)
    
    for s in subjects:
        path = cache_dir / f"{s}"
        path.mkdir(parents=True, exist_ok=True)
        
        extracted_features_path = path / f"{s}_extracted.npz"
        raw_features_path = path / f"{s}_raw.npz"

        extracted_exists = extracted_features_path.exists()
        if extracted_exists:            
            print(f"[CACHE] {s}_extracted exists")
        
        raw_exists = raw_features_path.exists()
        if raw_exists:
            print(f"[CACHE] {s}_raw exists")
                    
        if extracted_exists and raw_exists:
            continue

        df_s = merged_df.filter(pl.col("recording id") == s).collect(engine="streaming")

        if settings["use video min length"]:
            df_s = df_s.sort("timestamp [ns]").head(min_len)
        else:
            df_s = crop_video_percentage(
                df_s,
                settings["video percentage"],
                settings["video offset"],
            )       
        
        X, y = window_subject_raw(
            df_s,
            feature_cols,
            "label",
            settings
        )
        
        if settings["use engineered features"] and raw_exists is False:
            extracted_windows = pl.DataFrame(
                [extract_features(window) for window in X]
            ).to_numpy().astype(np.float32)
                        
            if np.isnan(extracted_windows).any():
                print("NaNs found before saving")
                print(np.where(np.isnan(extracted_windows)))
                
                nan_cols = [
                    col
                    for col, dtype in df_s.schema.items()
                    if dtype.is_float()
                    and df_s.select(pl.col(col).is_nan().any()).item()
                ]
                print(f"Cols with NaNs: {nan_cols}")
                raise ValueError("Feature extraction produced NaNs")

            print(f"[CACHE] {s}_extracted BUILD")
            np.savez_compressed(extracted_features_path, X=extracted_windows, y=y)

        if settings["use raw features"] and raw_exists is False:
            raw_windows = np.stack([
                window.drop(["fixation id", "blink id", "saccade id", "timestamp [ns]"]).to_numpy()
                for window in X
            ]).astype(np.float32)

            if np.isnan(raw_windows).any():
                print("NaNs found before saving")
                print(np.where(np.isnan(raw_windows)))
                
                nan_cols = [
                    col
                    for col, dtype in df_s.schema.items()
                    if dtype.is_float()
                    and df_s.select(pl.col(col).is_nan().any()).item()
                ]
                print(f"Cols with NaNs: {nan_cols}")
                raise ValueError("raw data have NaNs")

            print(f"[CACHE] {s}_raw BUILD")
            np.savez_compressed(raw_features_path, X=raw_windows, y=y)

In [11]:
def compute_features_per_subject(df: pl.DataFrame) -> pl.DataFrame:
    df = df.sort(["recording id", "timestamp [ns]"])
    
    df = df.with_columns([
        # --- diffs per recording ---
        pl.col("gaze x [px]").diff().over("recording id").alias("dx"),
        pl.col("gaze y [px]").diff().over("recording id").alias("dy"),
        pl.col("timestamp [ns]").diff().over("recording id").alias("dt"),
    ])

    df = df.with_columns([
        # --- velocity ---
        (
            (pl.col("dx")**2 + pl.col("dy")**2).sqrt()
            / pl.when(pl.col("dt") == 0).then(None).otherwise(pl.col("dt"))
        ).alias("velocity").fill_null(0),

        # --- pupil mean ---
        (
            (pl.col("pupil diameter left [mm]") + pl.col("pupil diameter right [mm]")) / 2
        ).alias("pupil mean"),
    ])

    # --- pupil normalization per recording ---
    df = df.with_columns([
        ((pl.col("pupil mean") - pl.col("pupil mean").mean().over("recording id"))
         / (pl.col("pupil mean").std().over("recording id") + 1e-6)
        ).alias("pupil norm")
    ])

    return df.drop(["dx", "dy", "dt"])

def extract_features(window: pl.DataFrame, width=None, height=None):

    def s(col):
        return window[col]

    features = {}

    vel = s("velocity").drop_nulls()

    # ------------------------
    # Velocity
    # ------------------------
    if len(vel) > 0:
        features.update({
            "vel mean": vel.mean(),
            "vel std": vel.std(),
            "vel max": vel.max(),
            "vel p25": vel.quantile(0.25),
            "vel p75": vel.quantile(0.75),
        })
    else:
        features.update({k: 0 for k in
            ["vel mean", "vel std", "vel max", "vel p25", "vel p75"]
        })


    # ------------------------
    # Event ratios
    # ------------------------
    features["saccade ratio"] = s("saccade id").is_not_null().mean()
    features["fixation ratio"] = s("fixation id").is_not_null().mean()
    features["blink ratio"] = s("blink id").is_not_null().mean()

    features["num fixations"] = s("fixation id").n_unique()
    features["num saccades"] = s("saccade id").n_unique()

    # ------------------------
    # Fixation duration
    # ------------------------
    fix = window.filter(pl.col("fixation id").is_not_null())

    if fix.height > 0:
        fix_dur = (
            fix.group_by("fixation id")
            .agg((pl.col("timestamp [ns]").max() - pl.col("timestamp [ns]").min()).alias("dur"))
            ["dur"]
        )

        features.update({
            "fix dur mean": fix_dur.mean(),
            "fix dur std": fix_dur.std(),
            "fix dur max": fix_dur.max(),
            "fix dur p25": fix_dur.quantile(0.25),
            "fix dur p75": fix_dur.quantile(0.75),
        })
    else:
        features.update({k: 0 for k in ["fix dur mean", "fix dur std", "fix dur max", "fix dur p25", "fix dur p75"]})

    # ------------------------
    # Saccade amplitude
    # ------------------------
    if window.height > 1:
        dx = s("gaze x [px]").diff().drop_nulls()
        dy = s("gaze y [px]").diff().drop_nulls()

        sac_amp = (dx**2 + dy**2).sqrt()

        features.update({
            "sac amp mean": sac_amp.mean(),
            "sac amp std": sac_amp.std(),
            "sac amp max": sac_amp.max(),
        })
    else:
        features.update({"sac amp mean": 0, "sac amp std": 0, "sac amp max": 0})

    # ------------------------
    # Event-specific velocity
    # ------------------------
    fix_vel = window.filter(pl.col("fixation id").is_not_null())["velocity"].drop_nulls()
    sac_vel = window.filter(pl.col("saccade id").is_not_null())["velocity"].drop_nulls()

    features["fix vel mean"] = fix_vel.mean() if len(fix_vel) else 0
    features["sac vel mean"] = sac_vel.mean() if len(sac_vel) else 0
    features["fix vel std"] = fix_vel.std() if len(fix_vel) else 0
    features["sac vel std"] = sac_vel.std() if len(sac_vel) else 0

    # ------------------------
    # Entropy
    # ------------------------
    hist, _, _ = np.histogram2d(s("gaze x [px]"), s("gaze y [px]"), bins=10)
    prob = hist / hist.sum() if hist.sum() > 0 else hist
    prob = prob[prob > 0]

    features["entropy"] = -np.sum(prob * np.log(prob)) if len(prob) else 0

    # ------------------------
    # Dispersion
    # ------------------------
    features["dispersion"] = (
        s("gaze x [px]").max() - s("gaze x [px]").min()
    ) + (
        s("gaze y [px]").max() - s("gaze y [px]").min()
    )

    # ------------------------
    # Gyro magnitude
    # ------------------------
    gyro_mag = (
        (s("gyro x [deg/s]")**2 +
         s("gyro y [deg/s]")**2 +
         s("gyro z [deg/s]")**2)
        .sqrt()
    )

    if len(gyro_mag):
        features.update({
            "gyro mag mean": gyro_mag.mean(),
            "gyro mag std": gyro_mag.std(),
            "gyro mag max": gyro_mag.max(),
            "head stability": 1 / (gyro_mag.std() + 1e-5),
        })
    else:
        features.update({"gyro mag mean": 0, "gyro mag std": 0, "gyro mag max": 0, "head stability": 0})

    # ------------------------
    # Pupil
    # ------------------------
    features["pupil mean"] = s("pupil norm").mean()
    features["pupil std"] = s("pupil norm").std()

    # ------------------------
    # Inside frame
    # ------------------------
    if width is not None and height is not None:
        inside = (
            (s("gaze x [px]") >= 0) &
            (s("gaze x [px]") <= width) &
            (s("gaze y [px]") >= 0) &
            (s("gaze y [px]") <= height)
        )
        features["inside ratio"] = inside.mean()

    features = {
        k: 0.0 if v is None or np.isnan(v) else float(v)
        for k, v in features.items()
    }
    
    return features

In [12]:
def window_subject_raw(
    df: pl.DataFrame,
    feature_cols,
    label_col,
    settings
):
    window_size = int(settings["window size"] * 1e9)
    stride = int(window_size * (1 - settings["overlap"]))
    target_length = settings["target length"]

    df = df.sort("timestamp [ns]")

    times = df["timestamp [ns]"]
    max_time = times[-1]
    min_time = times[0]

    X = []
    y = []

    start = min_time

    while start + window_size <= max_time:
        end = start + window_size

        window = df.filter(
            (pl.col("timestamp [ns]") >= start) &
            (pl.col("timestamp [ns]") < end)
        )

        if window.height < settings["min samples"]:
            start += stride
            continue        
        label = window[label_col][0]

        window_features = window.select(feature_cols)

        if window_features.height < settings["min samples"]:
            start += stride
            continue

        window_center = start + window_size / 2
        
        relative_time = (
            window_center - min_time
        ) / (
            max_time - min_time
        )

        window_features = window_features.with_columns(
            pl.lit(relative_time).alias("relative_time")
        )
        
        if window_features.height >= target_length:
            window_features = window_features.head(target_length)
        else:
            pad_size = target_length - window_features.height

            last_row = window_features.tail(1)

            padding = pl.concat([last_row] * pad_size, how="vertical")

            window_features = pl.concat([window_features, padding], how="vertical")

        X.append(window_features)
        y.append(label)

        start += stride

    return X, np.asarray(y)

In [13]:
def compute_subject_lengths(subjects, merged_df):
    lengths = {}

    for s in subjects:
        df_s = merged_df.filter(pl.col("recording id") == s).collect(engine="streaming")
        lengths[s] = df_s.height

    return lengths

In [14]:
settings = {
    "window size": 0.22,
    "overlap": 0.0,
    "target length": 45, # Sample rate is 200 Hz and window size is 0.22 so 45 is around 22% of 200
    "min samples": 5,
    "use video min length": False,
    "video percentage": 0.65,      # Fraction of the video to use (1.0 = 100%)
    "video offset": 0.0,          # Fraction to skip from the beginning
    "use engineered features": True,
    "use raw features": True,
}


subjects = (
    merged_df.select("recording id")
    .unique()
    .sort("recording id")
    .collect(engine="streaming")
    .to_series()
    .to_list()
)


cache_dir = get_cache_dir(training_data_path, settings, feature_cols)
print(f"Using cache dir: {cache_dir}")

merged_df = compute_features_per_subject(merged_df)
feature_cols.append("timestamp [ns]")
feature_cols.append('velocity')
feature_cols.append('pupil mean')
feature_cols.append('pupil norm')
    
build_cache(
    subjects=subjects,
    merged_df=merged_df,
    feature_cols=feature_cols,
    settings=settings,
    cache_dir=cache_dir
)

Using cache dir: ../../data/Final Training Data/Windowed Data/9a8e918cf3c537bbff7cfd9ed785b515
[CACHE] 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e_extracted BUILD
[CACHE] 0fc57dc8-de68-4b41-bf69-59a5a5f2d27e_raw BUILD
[CACHE] 10b8d8be-39de-43e7-9260-ba4cc724d4ae_extracted BUILD
[CACHE] 10b8d8be-39de-43e7-9260-ba4cc724d4ae_raw BUILD
[CACHE] 1279952d-14d4-4e77-9010-a000dd546bcd_extracted BUILD
[CACHE] 1279952d-14d4-4e77-9010-a000dd546bcd_raw BUILD
[CACHE] 1477b633-8ef3-4123-9ecc-f2ee90d13d8c_extracted BUILD
[CACHE] 1477b633-8ef3-4123-9ecc-f2ee90d13d8c_raw BUILD
[CACHE] 383b0fe8-654a-4f4f-84cd-375470069789_extracted BUILD
[CACHE] 383b0fe8-654a-4f4f-84cd-375470069789_raw BUILD
[CACHE] 3b1c507b-c290-4250-95c9-e21d6c52e3f2_extracted BUILD
[CACHE] 3b1c507b-c290-4250-95c9-e21d6c52e3f2_raw BUILD
[CACHE] 5bcd3401-e378-482f-b9c6-37edbad97d1a_extracted BUILD
[CACHE] 5bcd3401-e378-482f-b9c6-37edbad97d1a_raw BUILD
[CACHE] 63d66e1c-434f-4f23-8e74-abd8dedc0e43_extracted BUILD
[CACHE] 63d66e1c-434f-4f23-8e74-